# Transformations

This notebook covers essential data transformations in Spark.

## Learning Objectives

- Master map, flatMap, and filter operations
- Understand joins and their types
- Learn about aggregations
- Handle complex transformations

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split, concat, lit, when

spark = SparkSession.builder \
    .appName("Transformations") \
    .getOrCreate()

## 1. Map and FlatMap

These operations transform data element by element.

In [ ]:
# Create sample data
data = [("Hello World",), ("Apache Spark",), ("Python Programming",)]
df = spark.createDataFrame(data, ["text"])
df.show()

In [ ]:
# Map: one output per input
df.withColumn("word_count", 
    col("text").getItem(0)  # Just accessing first char as example
).show()

# Better example: length of text
from pyspark.sql.functions import length
df.withColumn("text_length", length(col("text"))).show()

In [ ]:
# FlatMap: multiple outputs per input (using explode)
# Split text into words and explode
words_df = df.withColumn("word", explode(split(col("text"), " ")))
words_df.show()

In [ ]:
# Count each word
words_df.groupBy("word").count().orderBy(col("count").desc()).show()

## 2. Filter Operations

Filtering removes rows that don't meet criteria.

In [ ]:
# Create sample data
employees = [
    ("Alice", 30, "Engineering", 85000),
    ("Bob", 25, "Marketing", 65000),
    ("Charlie", 35, "Engineering", 95000),
    ("Diana", 28, "Sales", 70000),
    ("Eve", 40, "Engineering", 105000),
]

df = spark.createDataFrame(employees, ["name", "age", "department", "salary"])

In [ ]:
# Simple filter
df.filter(col("salary") > 80000).show()

In [ ]:
# Multiple conditions (AND)
df.filter(
    (col("department") == "Engineering") & 
    (col("age") > 30)
).show()

In [ ]:
# Multiple conditions (OR)
df.filter(
    (col("department") == "Engineering") | 
    (col("salary") > 90000)
).show()

In [ ]:
# Filter with SQL-like syntax
df.filter("salary > 80000 AND age < 40").show()

## 3. Joins

Joins combine data from multiple DataFrames.

In [ ]:
# Create two DataFrames
employees = [
    (1, "Alice", 1),
    (2, "Bob", 2),
    (3, "Charlie", 1),
    (4, "Diana", 3),
]

departments = [
    (1, "Engineering"),
    (2, "Marketing"),
    (3, "Sales"),
    (4, "HR"),
]

emp_df = spark.createDataFrame(employees, ["emp_id", "name", "dept_id"])
dept_df = spark.createDataFrame(departments, ["dept_id", "dept_name"])

In [ ]:
# Inner join (default)
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "inner").show()

In [ ]:
# Left join
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "left").show()

In [ ]:
# Right join
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "right").show()

In [ ]:
# Full outer join
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "outer").show()

In [ ]:
# Join with column selection (avoid duplicate columns)
emp_df.join(dept_df, "dept_id").show()  # Using column name string for join key

## 4. Aggregations

Aggregations summarize data.

In [ ]:
# Create sample data
sales = [
    ("2024-01-01", "Electronics", 1000),
    ("2024-01-01", "Clothing", 500),
    ("2024-01-02", "Electronics", 1500),
    ("2024-01-02", "Clothing", 600),
    ("2024-01-03", "Electronics", 1200),
    ("2024-01-03", "Clothing", 700),
]

sales_df = spark.createDataFrame(sales, ["date", "category", "amount"])

In [ ]:
from pyspark.sql.functions import sum, avg, count, max, min

# Group by and aggregate
sales_df.groupBy("category").agg(
    sum("amount").alias("total_sales"),
    avg("amount").alias("avg_sales"),
    count("*").alias("transaction_count")
).show()

In [ ]:
# Multiple group by columns
sales_df.groupBy("date", "category").sum("amount").show()

In [ ]:
# Pivot table
sales_df.groupBy("date").pivot("category").sum("amount").show()

## 5. Working with Sample Data

In [ ]:
# Load sample data
try:
    users = spark.read.parquet("/opt/spark/data/users/small")
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    products = spark.read.parquet("/opt/spark/data/products/small")
    
    # Join users with orders
    user_orders = users.join(orders, "user_id", "inner")
    
    # Calculate total spending per user
    user_spending = user_orders.groupBy("user_id", "username").agg(
        sum("total_amount").alias("total_spent"),
        count("order_id").alias("order_count")
    )
    
    user_spending.orderBy(col("total_spent").desc()).show(10)
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 6. Exercises

In [ ]:
# Exercise 1: Find the top 3 users by order count
# Your code here:


In [ ]:
# Exercise 2: Calculate average order value per country
# Your code here:


In [ ]:
# Exercise 3: Find products that have never been ordered
# Your code here:


In [ ]:
# Clean up
spark.stop()